In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.utils import (
    apply_custom_style,
    load_dyst_samples,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "head_outputs")
os.makedirs(plot_save_dir, exist_ok=True)

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
split_name = "base40"
subsample_interval = 1

split_dir = os.path.join(DATA_DIR, split_name)
system_name = "Lorenz"
# ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

context_length = 512
context_start_time = 2000
context_end_time = context_start_time + context_length

# Prepare input series
sample_idx = 0
selected_dim = 0
dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :]).unsqueeze(0)
dyst_coords = dyst_coords[:, ::subsample_interval]
context = dyst_coords[:, context_start_time:context_end_time]

print(context.shape)

prediction_length = 512
future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
print(f"future_vals shape: {future_vals.shape}")

In [ ]:
# Plot the context series vs its index
context_np = context.squeeze(0).detach().cpu().numpy()
xs = np.arange(context_np.shape[-1])

plt.figure(figsize=(10, 3))
plt.plot(xs, context_np, linewidth=1.2)
plt.xlabel("context index")
plt.ylabel("value")
plt.title("Context series")
plt.tight_layout()

# Save and show
if "plot_save_dir" in globals():
    os.makedirs(plot_save_dir, exist_ok=True)
    plt.savefig(os.path.join(plot_save_dir, "context_series.png"), dpi=200)
plt.show()

In [ ]:
pipeline.add_v_attribution_hooks([(i, j) for j in range(num_heads) for i in range(num_layers)])
pipeline.add_head_attribution_hooks([(i, j) for j in range(num_heads) for i in range(num_layers)])

preds = pipeline.predict_with_full_outputs(
    context,
    prediction_length=prediction_length,
    output_attentions=True,
    output_hidden_states=True,
    return_dict_in_generate=True,
)

In [ ]:
def collect_ca_attention_scores(preds):
    dec_outs = preds[1]
    rollouts = len(dec_outs)

    num_layers = len(dec_outs[0].cross_attentions[0])
    num_samples, num_heads, _, context_len = dec_outs[0].cross_attentions[0][0].shape

    t = (rollouts - 1) * 64 + len(dec_outs[-1].cross_attentions)
    attention_scores = torch.zeros((t, num_layers, num_samples, num_heads, context_len))

    t_curr = 0
    for r in range(rollouts):
        for step in dec_outs[r].cross_attentions:
            for l in range(num_layers):
                attention_scores[t_curr, l] = step[l][:, :, -1, :]
            t_curr += 1

    return attention_scores

In [ ]:
sample = collect_ca_attention_scores(preds)[:, :, 0, :, :]
attnmaps = sample.detach().float().cpu().numpy()[:]  # [T, L, H, C]
T, L, H, C = attnmaps.shape

enc_hidden_states = preds[1][0]["encoder_hidden_states"][-1][0]  # [C, D]

In [ ]:
blocks = pipeline.model.model.decoder.block

d_model = pipeline.model.model.config.d_model
head_dim = d_model // num_heads
len(blocks), d_model, num_heads, head_dim

In [ ]:
def get_qkv(layer_idx: int, head_idx: int, cross_attn: bool = True):
    block = blocks[layer_idx]
    attn = block.layer[cross_attn].EncDecAttention
    Wq = attn.q.weight
    Wk = attn.k.weight
    Wv = attn.v.weight

    Wq = Wq.view(num_heads, head_dim, d_model)
    Wk = Wk.view(num_heads, head_dim, d_model)
    Wv = Wv.view(num_heads, head_dim, d_model)
    return Wq[head_idx], Wk[head_idx], Wv[head_idx]


def head_svd(layer_idx: int, head_idx: int, cross_attn: bool = True):
    Wq, Wk, Wv = get_qkv(layer_idx, head_idx, cross_attn)
    M = Wq.T @ Wk / np.sqrt(d_model)
    U, S, V = np.linalg.svd(M.float().detach().cpu().numpy())
    return U, S, V

In [ ]:
svds = [[head_svd(l, h) for h in range(num_heads)] for l in range(num_layers)]

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

for l in range(num_layers):
    for h in range(num_heads):
        _, S, _ = svds[l][h]
        kappa = S[0] / S[-1] if S[-1] != 0 else float("inf")
        ax.plot(S, label=f"L{l}, H{h} ($\\kappa$={kappa:.1e})", linewidth=1.3)

ax.set_title("SVD Spectra of All Layers and Heads", fontsize=15, fontweight="bold")
ax.set_xlabel("Singular Value Index", fontsize=13)
ax.set_ylabel("Singular Value", fontsize=13)
# ax.set_yscale("log")

handles, labels = ax.get_legend_handles_labels()
legend_cols = min(5, num_heads)
legend = ax.legend(
    handles,
    labels,
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize="small",
    ncol=legend_cols,
    borderaxespad=0.0,
    title="Layer/Head",
)
legend._legend_box.align = "left"
plt.subplots_adjust(right=0.78)  # leave space at right for legend

plt.tight_layout(rect=(0, 0, 0.78, 1))
plt.savefig(os.path.join(plot_save_dir, "svd_spectra.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
enc_hidden_heads = enc_hidden_states.detach().cpu().numpy()
key_norms = np.zeros((num_layers, num_heads))
for l in range(num_layers):
    for h in range(num_heads):
        V = svds[l][h][-1]
        h_tilde = V @ enc_hidden_heads.T  # [D, C]
        norms = h_tilde.T @ np.diag(svds[l][h][1]) @ h_tilde
        key_norms[l, h] = np.exp(0.5 * norms.diagonal()).mean()

plt.figure(figsize=(4, 4))
plt.title("key norms across layers and heads")
plt.imshow(key_norms, cmap="viridis")
plt.xticks(ticks=np.arange(num_heads), labels=[str(h) for h in range(num_heads)])
plt.yticks(ticks=np.arange(num_layers), labels=[str(l) for l in range(num_layers)])
plt.xlabel("Head index", fontweight="bold")
plt.ylabel("Layer index", fontweight="bold")
plt.colorbar()
plt.show()

In [ ]:
measure = np.zeros((num_layers, num_heads))
for l in range(num_layers):
    for h in range(num_heads):
        U, S, V = svds[l][h]

        S2 = S**2
        # measure[l, h] = S2.sum() / S2.max()
        measure[l, h] = S2.sum()
    measure[l] = measure[l] / measure[l].max()

plt.figure(figsize=(4, 4))
plt.title("Spectral energy")
plt.imshow(measure, cmap="viridis")
plt.xticks(ticks=np.arange(num_heads), labels=[str(h) for h in range(num_heads)])
plt.yticks(ticks=np.arange(num_layers), labels=[str(l) for l in range(num_layers)])
plt.xlabel("Head index", fontweight="bold")
plt.ylabel("Layer index", fontweight="bold")
plt.colorbar()
plt.show()

In [ ]:
# Plot attention heatmap for layer 0 with 4x3 grid of heads: x=timestep, y=context index
T, L, H, C = attnmaps.shape
assert L == 12 and H == 12, f"Expected 12 layers and 12 heads, got {L} and {H}"

layer = 8

fig, axes = plt.subplots(4, 3, figsize=(3 * 3, 4 * 2.5), sharex=True, sharey=True)
axes = axes.flatten()

for head in range(H):
    ax = axes[head]
    im = ax.imshow(attnmaps[:, layer, head, :].T, aspect="auto", origin="lower", extent=[0, T, 0, C], cmap="bone_r")
    ax.set_xlabel("timestep")
    ax.set_ylabel("context")
    ax.set_title(f"Layer {layer}, Head {head}")

    # Add colorbar for each subplot
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("attention score")

plt.tight_layout()

# Save and show
plt.savefig(os.path.join(plot_save_dir, f"attention_heatmap_L{layer}_all_heads.pdf"))
plt.show()

In [ ]:
T, L, H, C = attnmaps.shape

# Compute entropy: H = -sum_c p log p
eps = 1e-12
p_ctxt = attnmaps / (attnmaps.sum(axis=-1, keepdims=True) + eps)
ent_ctxt = (p_ctxt * np.log(p_ctxt + eps)).sum(axis=-1)  # [T, L, H]
attn_entropy = ent_ctxt.mean(axis=0) / np.log(C)

# Plot heatmap
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(attn_entropy, aspect="equal", cmap="viridis")
ax.set_xlabel("Head index", fontweight="bold")
ax.set_ylabel("Layer index", fontweight="bold")
ax.set_title("CA normalized negative entropy", fontweight="bold")
fig.colorbar(im, ax=ax, label="Entropy (nats)", shrink=0.8)
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir, "ca_entropy_heatmap_LxH.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
measure.shape, attn_entropy.shape

In [ ]:
plt.figure(figsize=(6, 4))
for l in range(num_layers):
    plt.scatter(measure[l], attn_entropy[l], label=f"Layer {l}", c=DEFAULT_COLORS[l])
plt.xlabel("Entropy of singular values")
plt.ylabel("Avg entropy of attention")
plt.legend(loc="upper left")
plt.show()


In [ ]:
from tsfm_lens.chronos_bolt.circuitlens import CircuitLensBolt

device = "cuda:0" if torch.cuda.is_available() else "cpu"
bolt = CircuitLensBolt.from_pretrained("amazon/chronos-bolt-small", device_map=device)
bolt.add_read_stream_hooks(list(range(bolt.num_layers)), stack_type="encoder")

print(f"num_layers: {bolt.num_layers}, num_heads: {bolt.num_heads}")

In [ ]:
from pathlib import Path

lorenz_data_dir = os.path.join(WORK_DIR, "lorenz_long/test")


def load_datasets(data_dir: Path) -> list[Path]:
    return sorted(Path(data_dir).glob("*.context.npy"))


data_path = load_datasets(lorenz_data_dir)[0]
data = np.load(data_path)

print(data.shape)

context_length = 1024
context_start_time = 343
context_end_time = context_start_time + context_length

# Prepare input series
seed = 8943
rng = np.random.default_rng(seed)
sample_idx = rng.choice(data.shape[0])
dyst_coords = torch.tensor(data[sample_idx, :].T).unsqueeze(0)
long_context = dyst_coords[..., context_start_time:context_end_time]

print(long_context.shape)

pred_length = 4096 + 1024
long_horizon = dyst_coords[..., context_end_time : context_end_time + pred_length]
print(f"long_horizon shape: {long_horizon.shape}")

In [ ]:
bolt_pred, bolt_out = bolt.predict_with_full_outputs(long_context[:, 0, :], prediction_length=pred_length)

In [ ]:
plt.plot(np.arange(context_start_time, context_start_time + context_length), long_context[0, 0, :], "k-.")
for i in range(bolt_pred.shape[1]):
    plt.plot(np.arange(context_end_time, context_end_time + pred_length), bolt_pred[0, i, :])
plt.show()

In [ ]:
rollouts = len(bolt_out)

num_heads, _, context_len = bolt_out[0].cross_attentions[0][0].shape

bolt_attn_scores = torch.zeros((rollouts, bolt.num_layers, bolt.num_heads, context_len))

for t in range(rollouts):
    step = bolt_out[t].cross_attentions
    for l in range(bolt.num_layers):
        bolt_attn_scores[t, l] = step[l][0, :, -1, -context_len:]

In [ ]:
bolt_attn_scores.shape

In [ ]:
def get_qkv_bolt(layer_idx: int, head_idx: int, cross_attn: bool = True):
    block = bolt.model.decoder.block[layer_idx]
    d_model = bolt.model.model_dim
    head_dim = d_model // bolt.num_heads
    attn = block.layer[cross_attn].EncDecAttention
    Wq = attn.q.weight
    Wk = attn.k.weight
    Wv = attn.v.weight

    Wq = Wq.view(bolt.num_heads, head_dim, d_model)
    Wk = Wk.view(bolt.num_heads, head_dim, d_model)
    Wv = Wv.view(bolt.num_heads, head_dim, d_model)
    return Wq[head_idx], Wk[head_idx], Wv[head_idx]


def head_svd_bolt(layer_idx: int, head_idx: int, cross_attn: bool = True):
    Wq, Wk, _ = get_qkv_bolt(layer_idx, head_idx, cross_attn)
    M = Wq.T @ Wk / np.sqrt(d_model)
    U, S, V = np.linalg.svd(M.float().detach().cpu().numpy())
    return U, S, V

In [ ]:
bolt_svds = [[head_svd_bolt(l, h) for h in range(bolt.num_heads)] for l in range(bolt.num_layers)]

In [ ]:
fig, axes = plt.subplots(
    bolt.num_layers, bolt.num_heads, figsize=(bolt.num_heads * 3, bolt.num_layers * 2.5), sharex=True, sharey=True
)
for l in range(bolt.num_layers):
    for h in range(bolt.num_heads):
        _, S, _ = bolt_svds[l][h]
        axes[l, h].plot(S, label=rf"$\kappa={S[0] / S[-1]:.2f}$")
        axes[l, h].set_title(f"Layer {l}, Head {h}")
        axes[l, h].legend()

plt.tight_layout()
plt.savefig(os.path.join(plot_save_dir, "bolt_svd_spectra.pdf"))
plt.show()

In [ ]:
step = 50  # block rollout step
key_norms = np.zeros((bolt.num_layers, bolt.num_heads))
for l in range(bolt.num_layers):
    enc_h = bolt.read_stream_outputs[l][step][0].detach().cpu().numpy()
    for h in range(bolt.num_heads):
        V = bolt_svds[l][h][-1]
        h_tilde = V @ enc_h.T  # [D, C]
        norms = h_tilde.T @ np.diag(bolt_svds[l][h][1]) @ h_tilde
        key_norms[l, h] = np.exp(0.5 * norms.diagonal()).mean()
    key_norms[l] = key_norms[l] / key_norms[l].max()

plt.figure(figsize=(4, 4))
plt.title(f"Encoder tilt (step {step})")
plt.imshow(key_norms, cmap="viridis")
plt.xticks(ticks=np.arange(bolt.num_heads), labels=[str(h) for h in range(bolt.num_heads)])
plt.yticks(ticks=np.arange(bolt.num_layers), labels=[str(l) for l in range(bolt.num_layers)])
plt.xlabel("Head index", fontweight="bold")
plt.ylabel("Layer index", fontweight="bold")
plt.colorbar()
plt.show()

In [ ]:
measure = np.zeros((bolt.num_layers, bolt.num_heads))
for l in range(bolt.num_layers):
    for h in range(bolt.num_heads):
        U, S, V = bolt_svds[l][h]

        S2 = S**2
        # measure[l, h] = S2.sum() / S2.max()
        measure[l, h] = S2.sum()
    measure[l] = measure[l] / measure[l].max()

plt.figure(figsize=(4, 4))
plt.title("Bolt spectral energy")
plt.imshow(measure, cmap="viridis")
plt.xticks(ticks=np.arange(bolt.num_heads), labels=[str(h) for h in range(bolt.num_heads)])
plt.yticks(ticks=np.arange(bolt.num_layers), labels=[str(l) for l in range(bolt.num_layers)])
plt.xlabel("Head index", fontweight="bold")
plt.ylabel("Layer index", fontweight="bold")
plt.colorbar()
plt.show()

In [ ]:
T, L, H, C = bolt_attn_scores.shape
assert L == 6 and H == 8, f"Expected 12 layers and 12 heads, got {L} and {H}"

layer = 3

fig, axes = plt.subplots(2, 4, figsize=(4 * 2.5, 2 * 2), sharex=True, sharey=True)
axes = axes.flatten()

for head in range(H):
    ax = axes[head]
    im = ax.imshow(
        bolt_attn_scores[:, layer, head, :].T, aspect="auto", origin="lower", extent=[0, T, 0, C], cmap="bone_r"
    )
    ax.set_ylabel("context")
    ax.set_xlabel("timestep")
    ax.set_title(f"Layer {layer}, Head {head}")

    # Add colorbar for each subplot
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("attention score")

plt.tight_layout()

# Save and show
plt.savefig(os.path.join(plot_save_dir, f"bolt_attention_heatmap_L{layer}_all_heads.pdf"))
plt.show()